# 05 · Orchestrating in Code

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

The SDK gives you two ways to decide what happens next. **Orchestrating via the
LLM**: give an agent tools and handoffs and let it choose. **Orchestrating via
code**: run agents from ordinary Python, and let `await`, `if`, `for` and
`asyncio.gather` decide the flow. This notebook is about the second. It's where
the SDK's "no graph, just Python" philosophy pays off.

**You will learn**
- When code should drive the flow, and when the model should
- **Chaining** agents, with a code **gate** in between
- **Routing** with a classifier agent and a plain `dict`
- **Parallel** agents with `asyncio.gather`, and dynamic **map-reduce**
- A guarded **refine loop** (writer + reviewer)
- What code orchestration does *not* give you: durability

**Prerequisites:** notebooks 01–04. *Optional background:* `00_CONCEPTS/03`
covers the same patterns without a framework.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import asyncio, time
from typing import Literal
from pydantic import BaseModel, Field
from agents import Agent, Runner
from common import get_model

model = get_model()

## 1 · Why orchestrate in code?

| | Via the LLM (tools, handoffs) | Via code (`await`, `if`, `gather`) |
|---|---|---|
| Who decides the next step | the model | you |
| Predictable order and branching | no | yes |
| Cost | extra model calls to decide | only the work itself |
| Testable without a model | hard | easy (it's Python) |
| Handles open-ended tasks | yes | only the paths you wrote |

Notebook 04 ended with a concrete case: a coordinator agent told to *edit, then
translate* ran both tools in parallel and translated the unedited draft. The
order mattered, and a prompt couldn't guarantee it. In code, it's two lines.

The rule of thumb: **use code for everything you can decide in advance, and let
the model decide only what genuinely needs judgement.**

## 2 · Chaining, with a gate

A chain runs agents one after another, each on the previous output. Between
steps you can put a **gate**: a plain code check that stops bad intermediate
results from flowing downstream.

```
 draft ──► outline agent ──► [gate: exactly 3 points?] ──► writer agent ──► translator agent
                                      │ no
                                      └──► stop early
```

In [ ]:
class Outline(BaseModel):
    points: list[str]

outliner = Agent(name='Outliner', model=model, output_type=Outline,
                 instructions='Give exactly 3 short outline points for the given topic. '
                              'Each point is a SEPARATE item in the points list.')
writer = Agent(name='Writer', model=model,
               instructions='Write one short paragraph (max 60 words) covering the points in order.')
translator = Agent(name='Translator', model=model,
                   instructions='Translate the text into Dutch. Reply with the translation only.')


async def announcement(topic: str) -> dict:
    outline = (await Runner.run(outliner, topic)).final_output
    if len(outline.points) != 3:                                          # the gate is CODE
        return {'error': f'outline had {len(outline.points)} points'}
    text = (await Runner.run(writer, '\n'.join(outline.points))).final_output
    dutch = (await Runner.run(translator, text)).final_output            # translates the EDITED text
    return {'outline': outline.points, 'english': text, 'dutch': dutch}


out = await announcement('Robot deliveries are paused on Monday for maintenance')
for key, value in out.items():
    print(f'{key:>8}: {value}')

If you ever see `error: outline had 1 points`, that's the gate doing its job.
While building this demo, the model sometimes packed all three points into one
string, until the instructions said explicitly that each point is a separate
item. The gate stopped the bad outline before two more model calls were wasted
on it.

The translator received exactly the writer's output, because the code passed
it along. There was no ambiguity to resolve and no extra model call to decide
the order.

## 3 · Routing with a classifier and a `dict`

To route, let a small agent **classify** (structured output, rubric in the
instructions, as in notebook 03), then choose the specialist with an ordinary
dictionary lookup. Low confidence falls back to a safe default, decided by code:

In [ ]:
class Route(BaseModel):
    department: Literal['billing', 'technical', 'general']
    confidence: float = Field(ge=0, le=1)

router = Agent(name='Router', model=model, output_type=Route, instructions=(
    'Route the support ticket. billing: charges, refunds, invoices. technical: bugs, errors, crashes. '
    'general: anything else. Give your confidence from 0 to 1.'))

specialists = {
    'billing':   Agent(name='Billing', model=model, instructions='You handle billing. Reply in 1 sentence.'),
    'technical': Agent(name='Tech', model=model, instructions='Give 1 troubleshooting step.'),
    'general':   Agent(name='General', model=model, instructions='Answer briefly and kindly.'),
}


async def handle(ticket: str) -> str:
    r = (await Runner.run(router, ticket)).final_output
    dept = r.department if r.confidence >= 0.6 else 'general'            # safe default in code
    reply = (await Runner.run(specialists[dept], ticket)).final_output
    return f'[{dept} {r.confidence:.2f}] {reply}'

for t in ['I was charged twice for order #88.', 'The export button throws error 500.', 'Do you have a newsletter?']:
    print(await handle(t), '\n')

## 4 · Parallel agents with `asyncio.gather`

Independent work should run at the same time. `Runner.run` is a coroutine, so
`asyncio.gather` runs several agents concurrently and returns their results in
order. Three analysts look at one idea; a fourth agent combines their views:

In [ ]:
ROLES = {
    'market analyst': 'Who would pay for this, and why? One sentence.',
    'engineer':       'What is the hardest technical problem? One sentence.',
    'risk officer':   'What is the biggest legal or safety risk? One sentence.',
}
analysts = [Agent(name=role, model=model, instructions=f'You are a {role}. {task}') for role, task in ROLES.items()]
judge = Agent(name='Judge', model=model, instructions='Given the notes, decide go or no-go in one sentence.')

idea = 'Robots that carry groceries up apartment stairs.'
t0 = time.perf_counter()
results = await asyncio.gather(*(Runner.run(a, idea) for a in analysts))      # all three at once
notes = '\n'.join(f'- {a.name}: {r.final_output}' for a, r in zip(analysts, results))
verdict = (await Runner.run(judge, notes)).final_output
print(notes, '\n\nverdict:', verdict, f'\n\n({time.perf_counter() - t0:.1f}s total)')

## 5 · Dynamic map-reduce

When the *number* of parallel tasks depends on the input, let a planner agent
produce a list (structured output), map a worker over it with `gather`, then
reduce with a writer. This is the orchestrator-workers pattern, written as
plain Python:

```
 question ──► planner (list of subtopics) ──► gather(worker(t1), worker(t2), …) ──► writer ──► answer
```

One detail matters: each worker gets **the original question as well as its
subtopic**. Given only "Safety concerns and regulations", a worker has no idea
*what* is being researched and drifts off-topic. (We hit exactly this while
building the LangGraph version of this demo.)

In [ ]:
class Plan(BaseModel):
    subtopics: list[str] = Field(description='2 to 4 distinct subtopics')

planner = Agent(name='Planner', model=model, output_type=Plan,
                instructions='Split the question into 2-4 distinct research subtopics.')
worker = Agent(name='Researcher', model=model,
               instructions='Write exactly two factual sentences on the given aspect of the question. No preamble.')
synthesiser = Agent(name='Writer', model=model,
                    instructions='Answer the question in 3 sentences using only the findings.')
MAX_SUBTOPICS = 4


async def research(question: str) -> str:
    plan = (await Runner.run(planner, question)).final_output
    subtopics = plan.subtopics[:MAX_SUBTOPICS]                            # bound the fan-out in code
    print('planned:', subtopics)
    findings = await asyncio.gather(*(
        Runner.run(worker, f'Question: {question}\nAspect: {s}') for s in subtopics))   # context + subtopic
    body = '\n'.join(f'## {s}\n{f.final_output}' for s, f in zip(subtopics, findings))
    return (await Runner.run(synthesiser, f'Question: {question}\n\nFindings:\n{body}')).final_output

print(await research('Should a city allow sidewalk delivery robots?'))

Notice `plan.subtopics[:MAX_SUBTOPICS]`: a planner could return 30 subtopics,
so the fan-out is bounded **in code**. If you use a cloud API, also limit
concurrency (e.g. with an `asyncio.Semaphore`) to respect rate limits.

## 6 · A guarded refine loop

A writer drafts, a reviewer judges, and failing drafts go back with feedback.
As in every loop: **a hard cap** on rounds, and a **graceful exit** that flags
the result instead of pretending it passed. The reviewer checks exact things in
code first and asks the model only the judgement question.

In [ ]:
class Review(BaseModel):
    passed: bool
    feedback: str

BRIEF = 'Courier C2 robots get a new battery pack with 30% more range.'
drafter = Agent(name='Drafter', model=model, instructions=(
    'Write a one-sentence release note for the brief. Use ONLY facts from the brief; never invent numbers. '
    'Reply with the sentence only.'))
reviewer = Agent(name='Reviewer', model=model, output_type=Review, instructions=(
    'Pass the release note if it is professional, free of hype, and states only facts from the brief. '
    'Do not ask for details the brief does not contain.'))
MAX_ROUNDS = 3


async def release_note(brief: str) -> tuple[str, str]:
    draft, feedback = '', ''
    for round_ in range(1, MAX_ROUNDS + 1):
        prompt = brief if not feedback else f'{brief}\nPrevious: {draft}\nFix: {feedback}'
        draft = (await Runner.run(drafter, prompt)).final_output.strip()
        problems = []
        if len(draft.split()) > 25:
            problems.append(f'max 25 words (has {len(draft.split())})')
        if 'battery' not in draft.lower():
            problems.append('must mention the battery')
        if not problems:
            review = (await Runner.run(reviewer, f'Brief: {brief}\nRelease note: {draft}')).final_output
            if not review.passed:
                problems.append(review.feedback)
        print(f'round {round_}: {draft}\n   → {"PASS" if not problems else "; ".join(problems)}')
        if not problems:
            return draft, 'approved'
        feedback = '; '.join(problems)
    return draft, f'FLAGGED for human review after {MAX_ROUNDS} rounds'

note, status = await release_note(BRIEF)
print('\nstatus:', status)

## 7 · What code orchestration doesn't give you

Everything above lives in **one Python process's memory**. If the process dies
halfway through `research()`, the planner's output and every finished worker
result are gone, and a rerun pays for all of them again. There are no
checkpoints between the `await`s.

For short workflows that's fine. For long or expensive ones you add durability
yourself: save each step's result (to a file or database) and skip steps that
are already done on restart. Notebook 09 explains what the SDK *does* persist
(sessions, paused runs), and the project adds step checkpoints by hand. Graph
frameworks like LangGraph, or durable-execution engines like Temporal, give you
this automatically; that's one of the main reasons to choose them.

## 8 · Choosing: code or LLM?

| Situation | Orchestrate via |
|---|---|
| Steps and their order are known | code (chain) |
| Inputs fall into known categories | code (classifier agent + `dict`) |
| Independent parts / several opinions | code (`gather`) |
| Subtasks depend on the input | code (planner agent + `gather`) |
| Refine until good enough | code (loop with a cap) |
| The path can't be predicted (open-ended research, debugging) | the LLM (tools, handoffs) |
| A conversation should move to a specialist | the LLM (handoffs, notebook 06) |

Mixing is normal: a code workflow whose steps are agents, some of which use tools freely.

## 9 · Common mistakes

- **Asking an agent to run a fixed procedure.** If you know the steps, write them in code.
- **Awaiting independent agents one by one.** Use `asyncio.gather`.
- **Unbounded fan-out.** Cap the planner's list in code, and limit concurrency.
- **Workers without context.** Send the question *and* the subtask.
- **Loops without a cap**, or ones that return a failing result as if it passed.
- **Assuming long workflows survive a crash.** They don't, unless you save progress.

## 10 · Check yourself

<details><summary><b>1.</b> Why did the chain in §2 translate the right text when the coordinator in notebook 04 didn't?</summary>

In code, the translator is called after the writer finishes and receives the writer's output as its input, so the order is guaranteed. The coordinator agent batched both tool calls in one turn, so they ran in parallel on the original text.
</details>

<details><summary><b>2.</b> How do you run three independent agents concurrently, and what do you get back?</summary>

`await asyncio.gather(*(Runner.run(a, x) for a in agents))`. It returns the three `RunResult`s in the same order as the agents, after all have finished.
</details>

<details><summary><b>3.</b> In map-reduce, why cap the planner's list in code and give workers the question?</summary>

The cap bounds cost and concurrency if the planner returns too many items. The question gives each worker the context it needs; a bare subtopic invites off-topic answers.
</details>

<details><summary><b>4.</b> What happens to a code-orchestrated workflow if the process crashes halfway?</summary>

Everything in memory is lost: finished steps must be re-run (and re-paid) on restart, unless you saved each step's result and skip completed ones. The SDK doesn't checkpoint between `await`s.
</details>

<details><summary><b>5.</b> Give one case where the LLM should decide the flow instead of code.</summary>

When the steps can't be predicted: open-ended research, debugging an unknown problem, or a conversation that should move to whichever specialist fits.
</details>

## Takeaway

With the Agents SDK, a workflow is **Python calling agents**: `await` for order,
`if`/`dict` for routing, `asyncio.gather` for parallelism, `for` with a cap for
refinement. It's predictable, cheap and testable. Leave decisions to the model
only where judgement is needed, and remember that nothing survives a crash
unless you save it.

Next → `06_handoffs_and_multi_agent.ipynb`: letting agents pass the conversation
to each other.